# 🍽️ EDA — Gastronomía de Euskadi
**Fuente:** OpenData Euskadi  
**Dataset:** Establecimientos gastronómicos registrados en Turismo Euskadi

Este análisis exploratorio cubre:
- Distribución por tipo de establecimiento, provincia y entorno
- Presencia de sellos de calidad (Michelin, Repsol)
- Análisis de valoraciones y reseñas
- Distribución geográfica
- Completitud del dataset (valores nulos)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# Paleta de colores identitaria del País Vasco (verdes y rojos txuri-gorri)
PALETTE = ['#C0392B', '#2E86C1', '#27AE60', '#F39C12', '#8E44AD', '#16A085', '#E67E22', '#2C3E50']
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print('✅ Librerías cargadas')

## 1. Carga y primera inspección del dataset

In [ ]:
df = pd.read_csv('gastronomia.csv', sep=None, engine='python', index_col=0)

print(f'📊 Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'📋 Columnas: {df.columns.tolist()}')
df.head(3)

### Tipos de datos y resumen estadístico

In [ ]:
df.info()

In [ ]:
df[['valoracion', 'num_resenas', 'Michelin', 'Repsol']].describe().round(2)

> **Observación:** El dataset contiene 490 establecimientos gastronómicos. Las variables numéricas relevantes son `valoracion` (nota media de Google) y `num_resenas`. Los campos `Michelin` y `Repsol` son indicadores binarios (0/1).

## 2. Calidad del dataset — Valores nulos

In [ ]:
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(1)
nulos_df = pd.DataFrame({'Nulos': nulos, 'Porcentaje (%)': nulos_pct})
nulos_df = nulos_df[nulos_df['Nulos'] > 0].sort_values('Porcentaje (%)', ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(nulos_df.index, nulos_df['Porcentaje (%)'], color=PALETTE[0], edgecolor='white')
ax.bar_label(bars, fmt='%.1f%%', padding=4, fontsize=9)
ax.set_xlabel('% de valores nulos')
ax.set_title('Completitud del dataset — Campos con valores nulos', fontweight='bold')
ax.set_xlim(0, 75)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(nulos_df.to_string())

> **Observación:** Los campos más incompletos son `Sello` (60%) y `Nivel precio` (55%), lo que indica que muchos establecimientos no han sido categorizados en estas dimensiones. El campo `Email` también tiene un 20% de valores ausentes. Las coordenadas geográficas (`lat`, `lon`) y los indicadores de calidad (`Michelin`, `Repsol`) están **completamente rellenos**, lo que garantiza análisis fiables en esas dimensiones.

## 3. Distribución por tipo de establecimiento

In [ ]:
tipo_counts = df['Tipo de lugar'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Barplot
bars = axes[0].bar(tipo_counts.index, tipo_counts.values, color=PALETTE[:len(tipo_counts)], edgecolor='white')
axes[0].bar_label(bars, padding=3, fontsize=10)
axes[0].set_xlabel('Tipo de establecimiento')
axes[0].set_ylabel('Número de locales')
axes[0].set_title('Establecimientos por tipo', fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)

# Pie chart
wedges, texts, autotexts = axes[1].pie(
    tipo_counts.values, labels=tipo_counts.index, autopct='%1.1f%%',
    colors=PALETTE[:len(tipo_counts)], startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
for at in autotexts:
    at.set_fontsize(9)
axes[1].set_title('Proporción por tipo de lugar', fontweight='bold')

plt.tight_layout()
plt.show()

> **Observación:** Los **Restaurantes** dominan ampliamente con 279 registros (57%), seguidos de **Bodegas** (13%) y **Sidrerías** (9%), reflejo de la cultura gastronómica vasca. La presencia notable de bodegas de Txakoli (33) evidencia la importancia de la denominación de origen local. Las tiendas y queserías completan el panorama del producto artesanal.

## 4. Distribución geográfica por provincia

In [ ]:
prov_tipo = df.groupby(['Provincia', 'Tipo de lugar']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(11, 5))
prov_tipo.plot(kind='bar', ax=ax, color=PALETTE[:len(prov_tipo.columns)], edgecolor='white', width=0.7)
ax.set_xlabel('Provincia')
ax.set_ylabel('Número de establecimientos')
ax.set_title('Distribución de establecimientos por provincia y tipo', fontweight='bold')
ax.legend(title='Tipo de lugar', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

print('\nTotal por provincia:')
print(df['Provincia'].value_counts().to_string())

> **Observación:** **Guipúzcoa** lidera con 216 establecimientos, seguida de **Vizcaya** (142) y **Álava** (132). Álava destaca por sus Bodegas de Rioja Alavesa, mientras que Guipúzcoa concentra la mayor oferta de restaurantes y sidrerías, coherente con su reconocida tradición culinaria y la densidad urbana de San Sebastián / Donostia.

## 5. Entornos turísticos

In [ ]:
# Explotar entornos múltiples (separados por coma)
entorno_exploded = df['Entorno'].dropna().str.split(',').explode().str.strip()
entorno_counts = entorno_exploded.value_counts()

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(entorno_counts.index, entorno_counts.values, color=PALETTE[1], edgecolor='white')
ax.bar_label(bars, padding=4, fontsize=9)
ax.set_xlabel('Número de establecimientos')
ax.set_title('Establecimientos por entorno turístico', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

> **Observación:** **Montes y Valles vascos** es el entorno con mayor presencia gastronómica (190 menciones), seguido de la **Costa Vasca** (104) y **Rioja Alavesa** (80), zona vinícola por excelencia. San Sebastián y Bilbao, como capitales urbanas, también concentran una parte relevante de la oferta de restauración de calidad.

## 6. Sellos de calidad: Michelin y Repsol

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# Michelin
michelin_counts = df['Michelin'].value_counts().rename({0: 'Sin estrella', 1: 'Con estrella ⭐'})
axes[0].pie(michelin_counts.values, labels=michelin_counts.index, autopct='%1.1f%%',
            colors=['#BDC3C7', '#F1C40F'], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Estrella Michelin', fontweight='bold')

# Repsol
repsol_counts = df['Repsol'].value_counts().rename({0: 'Sin sol', 1: 'Con sol 🌞'})
axes[1].pie(repsol_counts.values, labels=repsol_counts.index, autopct='%1.1f%%',
            colors=['#BDC3C7', '#E67E22'], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Sol Repsol', fontweight='bold')

# Michelin por provincia
mich_prov = df[df['Michelin'] == 1]['Provincia'].value_counts()
axes[2].bar(mich_prov.index, mich_prov.values, color=PALETTE[3], edgecolor='white')
axes[2].bar_label(axes[2].containers[0], padding=3)
axes[2].set_title('Estrellas Michelin por provincia', fontweight='bold')
axes[2].set_ylabel('Número de estrellas')

plt.suptitle('Sellos de calidad gastronómica', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Establecimientos con estrella Michelin: {df["Michelin"].sum()} ({df["Michelin"].mean()*100:.1f}%)')
print(f'Establecimientos con sol Repsol: {df["Repsol"].sum()} ({df["Repsol"].mean()*100:.1f}%)')

> **Observación:** Solo el **4.5% de los establecimientos** cuenta con estrella Michelin (22 locales) y el **12.2% con sol Repsol** (60 locales). **Guipúzcoa** concentra la mayoría de las estrellas Michelin, coherente con la fama internacional de San Sebastián como capital gastronómica mundial por densidad de estrellas per cápita.

## 7. Nivel de precio

In [ ]:
precio_order = ['Moderado', 'Caro', 'Muy caro']
precio_counts = df['Nivel precio'].value_counts().reindex(precio_order).dropna()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Barplot de nivel de precio
colors_precio = ['#27AE60', '#E67E22', '#C0392B']
bars = axes[0].bar(precio_counts.index, precio_counts.values, color=colors_precio, edgecolor='white')
axes[0].bar_label(bars, padding=3)
axes[0].set_ylabel('Número de establecimientos')
axes[0].set_title('Nivel de precio (sobre 220 con dato)', fontweight='bold')

# Valoración media por nivel de precio
val_precio = df.groupby('Nivel precio')['valoracion'].mean().reindex(precio_order).dropna()
axes[1].bar(val_precio.index, val_precio.values, color=colors_precio, edgecolor='white')
axes[1].set_ylim(3.5, 5)
axes[1].set_ylabel('Valoración media (Google)')
axes[1].set_title('Valoración media por nivel de precio', fontweight='bold')
for i, (k, v) in enumerate(val_precio.items()):
    axes[1].text(i, v + 0.01, f'{v:.2f}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

> **Observación:** Entre los locales con nivel de precio registrado, predominan los de precio **Moderado** (73%). Curiosamente, la valoración media sube con el precio: los establecimientos **Muy caros** alcanzan una nota media mayor, lo que sugiere que el alto precio está asociado a mayor calidad percibida y experiencias diferenciales (restaurantes de alta cocina, estrellados).

## 8. Análisis de valoraciones y reseñas

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Distribución de valoraciones
axes[0].hist(df['valoracion'], bins=20, color=PALETTE[2], edgecolor='white')
axes[0].axvline(df['valoracion'].mean(), color='red', linestyle='--', label=f'Media: {df["valoracion"].mean():.2f}')
axes[0].axvline(df['valoracion'].median(), color='orange', linestyle='--', label=f'Mediana: {df["valoracion"].median():.2f}')
axes[0].set_xlabel('Valoración')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de valoraciones', fontweight='bold')
axes[0].legend(fontsize=9)

# Distribución de reseñas (log scale)
axes[1].hist(df['num_resenas'].dropna(), bins=30, color=PALETTE[1], edgecolor='white')
axes[1].set_xlabel('Número de reseñas')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución de número de reseñas', fontweight='bold')
axes[1].set_yscale('log')

# Valoración media por tipo de lugar
val_tipo = df.groupby('Tipo de lugar')['valoracion'].mean().sort_values(ascending=True)
bars = axes[2].barh(val_tipo.index, val_tipo.values, color=PALETTE[:len(val_tipo)], edgecolor='white')
axes[2].set_xlim(3.5, 4.8)
axes[2].set_xlabel('Valoración media')
axes[2].set_title('Valoración media por tipo de lugar', fontweight='bold')
for bar, val in zip(bars, val_tipo.values):
    axes[2].text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f'Valoración media global: {df["valoracion"].mean():.2f}')
print(f'Mediana de reseñas: {df["num_resenas"].median():.0f}')
print(f'Máximo de reseñas: {df["num_resenas"].max():.0f} — {df.loc[df["num_resenas"].idxmax(), "Nombre"]}')

> **Observación:** Las valoraciones se concentran entre **4.0 y 4.8**, con una media de ~4.3, lo cual refleja una oferta gastronómica de alta calidad percibida. La distribución de reseñas es muy asimétrica (cola larga a la derecha): la mayoría tiene pocas reseñas, pero algunos establecimientos acumulan miles. Las **Queserías** tienen la valoración media más alta, posiblemente por ser establecimientos especializados con clientela muy fiel.

## 9. Valoración vs. Michelin y Repsol

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, col, label, color_on, color_off in [
    (axes[0], 'Michelin', 'Estrella Michelin', '#F1C40F', '#BDC3C7'),
    (axes[1], 'Repsol', 'Sol Repsol', '#E67E22', '#BDC3C7')
]:
    data_no = df[df[col] == 0]['valoracion']
    data_si = df[df[col] == 1]['valoracion']
    ax.boxplot([data_no, data_si], labels=[f'Sin {label}', f'Con {label}'],
               patch_artist=True,
               boxprops=dict(facecolor=color_off),
               medianprops=dict(color='black', linewidth=2))
    ax.containers[0].patches[1].set_facecolor(color_on)
    ax.set_ylabel('Valoración')
    ax.set_title(f'Valoración según {label}', fontweight='bold')
    ax.set_ylim(2, 5.5)
    mean_no = data_no.mean()
    mean_si = data_si.mean()
    ax.text(1, 5.2, f'μ={mean_no:.2f}', ha='center', fontsize=10, color='gray')
    ax.text(2, 5.2, f'μ={mean_si:.2f}', ha='center', fontsize=10, color='darkorange')

plt.tight_layout()
plt.show()

> **Observación:** Los establecimientos con sello **Michelin o Repsol** presentan valoraciones medias ligeramente superiores y menor variabilidad (distribución más concentrada). Esto confirma que los sellos de calidad oficial están alineados con la percepción positiva de los usuarios en plataformas digitales.

## 10. Mapa de calor: correlación entre variables numéricas

In [ ]:
num_cols = ['valoracion', 'num_resenas', 'Michelin', 'Repsol', 'Calidad', 'Patrocinado']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            mask=mask, ax=ax, linewidths=0.5, vmin=-1, vmax=1)
ax.set_title('Correlación entre variables numéricas', fontweight='bold')
plt.tight_layout()
plt.show()

> **Observación:** Existe una correlación positiva moderada entre tener **Sol Repsol** y **Estrella Michelin** (los mejores restaurantes suelen acumular ambos reconocimientos). La correlación entre `valoracion` y `num_resenas` es débil, indicando que la popularidad (volumen de reseñas) no necesariamente va de la mano de la calidad percibida.

## 11. Distribución geográfica (scatter plot)

In [ ]:
prov_colors = {'Guipúzcoa': '#C0392B', 'Vizcaya': '#2E86C1', 'Álava': '#27AE60'}

fig, ax = plt.subplots(figsize=(12, 6))

for prov, color in prov_colors.items():
    subset = df[df['Provincia'] == prov]
    ax.scatter(subset['lon'], subset['lat'], c=color, alpha=0.6, s=30, label=prov)

# Resaltar Michelin
mich = df[df['Michelin'] == 1]
ax.scatter(mich['lon'], mich['lat'], c='gold', edgecolors='black', s=100, zorder=5, marker='*', label='Estrella Michelin')

ax.set_xlabel('Longitud')
ax.set_ylabel('Latitud')
ax.set_title('Distribución geográfica de establecimientos gastronómicos en Euskadi', fontweight='bold')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

> **Observación:** El mapa de dispersión revela los clústeres urbanos de **Bilbao** (Vizcaya, costa oeste) y **San Sebastián** (Guipúzcoa, costa este), así como la dispersión de establecimientos en el interior de Álava (Rioja Alavesa al sur). Las estrellas Michelin (⭐ doradas) se concentran principalmente en el eje costero y las capitales.

## 12. Top 10 — Establecimientos mejor valorados (con más de 100 reseñas)

In [ ]:
top10 = (
    df[df['num_resenas'] >= 100]
    .sort_values('valoracion', ascending=False)
    .head(10)[['Nombre', 'Municipio', 'Tipo de lugar', 'valoracion', 'num_resenas', 'Michelin', 'Repsol']]
    .reset_index(drop=True)
)
top10.index += 1
top10.columns = ['Nombre', 'Municipio', 'Tipo', 'Valoración', 'Reseñas', 'Michelin⭐', 'Repsol🌞']

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(top10['Nombre'][::-1], top10['Valoración'][::-1], color=PALETTE[2], edgecolor='white')
ax.set_xlim(4.5, 5.1)
ax.set_xlabel('Valoración media')
ax.set_title('Top 10 establecimientos mejor valorados (≥100 reseñas)', fontweight='bold')
for bar, val in zip(bars, top10['Valoración'][::-1]):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.1f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print(top10.to_string())

> **Observación:** El filtro de 100+ reseñas garantiza que el ranking sea estadísticamente robusto y no esté dominado por establecimientos con muy pocas valoraciones. Los mejores establecimientos alcanzan notas de 4.8–5.0, una cifra excepcionalmente alta.

---
## 📝 Resumen ejecutivo

| Dimensión | Hallazgo clave |
|---|---|
| **Volumen** | 490 establecimientos registrados |
| **Tipo dominante** | Restaurantes (57%), seguidos de Bodegas y Sidrerías |
| **Provincia líder** | Guipúzcoa (44%), especialmente en alta cocina |
| **Calidad media** | Valoración media de ~4.3/5 — oferta de alta calidad |
| **Estrellas Michelin** | 22 restaurantes (4.5%), concentrados en Guipúzcoa |
| **Sol Repsol** | 60 establecimientos (12.2%) |
| **Dato incompleto** | Nivel precio y Sello faltan en >50% de registros |
| **Geografía** | Clústeres en San Sebastián, Bilbao y Rioja Alavesa |